Alerta de ordens pendentes de criação de KPI:
ordens pendentes, quantidade de itens e informações de cliente para envio de alerta por e-mail e telefone de quem o cadastro completo

In [0]:
bronze_path = '/Volumes/bikestore/bronze/bronze_volumes'
silver_path = '/Volumes/bikestore/silver/silver_volumes'
gold_path = '/Volumes/bikestore/gold/gold_volumes'

In [0]:
silver_tables = {
    'tmp_silver_customers' : f'{silver_path}/customer',
    'tmp_silver_orders' : f'{silver_path}/orders'
}

for view_name, path in silver_tables.items():
    (spark.read.format('delta')
        .load(path)
        .createOrReplaceTempView(view_name))

In [0]:
df_orders_pending_gold = spark.sql("""
WITH pending AS (
SELECT
    customer_id,
    order_date,
    sum(quantity) as quantity,
    store_name
FROM tmp_silver_orders
WHERE lower(status) = 'pending'
GROUP BY customer_id, store_name, order_date
)

SELECT 
    p.*,
    customer.first_name AS fist_name_customer,
    customer.email,
    customer.phone
FROM pending AS p
LEFT JOIN tmp_silver_customers AS customer ON p.customer_id = customer.customer_id
WHERE customer.email IS NOT NULL
AND customer.phone IS NOT NULL
""")

df_orders_pending_gold.write.mode("overwrite").format("delta").option(
    "mergeSchema", "true"
).save(gold_path + "/orders_pending")

In [0]:
%sql
-- Criando uma tabela física
CREATE TABLE IF NOT EXISTS bikestore.gold.orders_pending
USING DELTA
AS
SELECT *
FROM delta.`/Volumes/bikestore/gold/gold_volumes/orders_pending`;